# 05 — Model Explainability (SHAP)

**Why explainability matters:**
A black-box model that says 'your price should be €250' is not very useful.
SHAP tells you WHY — which features pushed the price up or down and by how much.

**What SHAP does:**
For each listing, SHAP assigns a value to each feature showing its contribution.
Positive SHAP value = pushed price UP
Negative SHAP value = pushed price DOWN

**Example for one listing:**
- Base (average) price: €150
- neighbourhood_price_encoded: +€45 (premium area)
- is_entire_home: +€30 (whole apartment)
- accommodates: +€15 (sleeps 6)
- minimum_nights: -€8 (requires 7 nights)
- **Predicted: €232**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import json
import shap
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

BLUE  = '#185FA5'
TEAL  = '#1D9E75'
RED   = '#A32D2D'
REPORTS = Path('../../reports')

model_df = pd.read_parquet('../data/model_ready.parquet')
with open('../data/feature_list.json') as f:
    FEATURES = json.load(f)

X = model_df[FEATURES]
y = model_df['log_price']

# Load XGBoost (best model for SHAP — native TreeExplainer support)
with open('../models/xgboost.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

print(f'Dataset: {X.shape[0]:,} rows x {X.shape[1]} features')
print('XGBoost model loaded')

In [ ]:
# Compute SHAP values
# Use sample of 1,500 for reasonable speed
sample_size = min(1500, len(X))
X_sample    = X.sample(sample_size, random_state=42)

print(f'Computing SHAP values for {sample_size} listings...')
print('(Usually takes 30-60 seconds)')

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_sample)

print('Done!')
print(f'SHAP values shape: {shap_values.shape}  ({shap_values.shape[0]} listings x {shap_values.shape[1]} features)')

In [ ]:
# Feature importance — mean absolute SHAP value
importance = pd.DataFrame({
    'feature'  : FEATURES,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)

print('Feature importance (SHAP) — Top 15:')
print('-' * 50)
for i, (_, row) in enumerate(importance.head(15).iterrows(), 1):
    bar = '|' * int(row['importance'] * 200)
    print(f'  {i:2d}. {row["feature"]:<35}: {row["importance"]:.4f}  {bar}')

In [ ]:
# Chart 1: Feature importance bar chart
fig, ax = plt.subplots(figsize=(10, 7))

top15  = importance.head(15)
colors = [TEAL if i < 3 else (BLUE if i < 8 else '#888780')
          for i in range(len(top15))]

bars = ax.barh(top15['feature'][::-1], top15['importance'][::-1],
               color=colors[::-1], edgecolor='white', height=0.7)

for bar, val in zip(bars, top15['importance'][::-1]):
    ax.text(bar.get_width()+0.0003,
            bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Mean |SHAP value| — average impact on log(price)', fontsize=11)
ax.set_title('Top 15 Features Driving Amsterdam Airbnb Prices\n'
             '(teal = top 3 most important)',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(REPORTS / 'fig_ml_shap_importance.png', dpi=130, bbox_inches='tight')
plt.show()

top3 = importance.head(3)['feature'].tolist()
print(f"""
Top 3 price drivers:
  1. {top3[0]}
  2. {top3[1]}
  3. {top3[2]}

These three features alone explain the majority of price variation.
This confirms what EDA showed — location and room type are the biggest levers.
""")

In [ ]:
# Chart 2: SHAP beeswarm (summary) plot
# Each dot = one listing
# X position = how much that feature pushed price up or down
# Color = high value (pink) vs low value (blue) of that feature

top10_features = importance.head(10)['feature'].tolist()
top10_idx      = [FEATURES.index(f) for f in top10_features]

fig = plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values[:, top10_idx],
    X_sample[top10_features],
    plot_type='dot',
    show=False,
    max_display=10
)
plt.title('SHAP Beeswarm — Direction and Magnitude of Feature Impact\n'
          'Pink = high feature value, Blue = low feature value',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'fig_ml_shap_beeswarm.png', dpi=130, bbox_inches='tight')
plt.show()

print("""
How to read this chart:

  Each ROW = one feature
  Each DOT = one listing
  X position = SHAP value (right = pushed price UP, left = pushed price DOWN)
  Color = PINK means the listing has a HIGH value for that feature
          BLUE means the listing has a LOW value for that feature

Example — neighbourhood_price_encoded (top feature):
  Pink dots far right = listings in expensive neighbourhoods are
  predicted MUCH higher. Makes sense.
  Blue dots far left = listings in cheap neighbourhoods predicted lower.

Example — is_entire_home:
  Pink (=1, entire home) dots on right = entire homes price higher.
  Blue (=0, not entire) dots on left = non-entire pushes price down.
""")

In [ ]:
# Chart 3: SHAP dependence plot for top feature
# Shows exactly HOW that feature affects price across its range

top_feature = importance.iloc[0]['feature']
top_idx     = FEATURES.index(top_feature)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('How Top 2 Features Affect Price — SHAP Dependence',
             fontsize=13, fontweight='bold')

for ax, feat in zip(axes, importance.head(2)['feature'].tolist()):
    feat_idx = FEATURES.index(feat)
    x_vals   = X_sample[feat].values
    s_vals   = shap_values[:, feat_idx]

    sc = ax.scatter(x_vals, s_vals, alpha=0.3, s=15,
                    c=s_vals, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
    ax.set_xlabel(feat.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('SHAP value\n(positive = pushes price up)', fontsize=10)
    ax.set_title(f'Effect of {feat.replace("_"," ")} on price prediction')
    plt.colorbar(sc, ax=ax, label='SHAP value')

plt.tight_layout()
plt.savefig(REPORTS / 'fig_ml_shap_dependence.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 4: Explain a single listing prediction
# Pick a typical listing and show exactly why the model predicted that price

# Pick a listing near the median price
median_price = model_df['price'].median()
sample_listing_idx = (
    (model_df['price'] - median_price).abs().argsort().iloc[0]
)
listing = X_sample.iloc[0:1]  # use first sample listing

listing_shap = explainer.shap_values(listing)
base_value   = explainer.expected_value
pred_log     = xgb_model.predict(listing)[0]
pred_price   = np.expm1(pred_log)

# Show top contributing features for this listing
contrib = pd.DataFrame({
    'feature'    : FEATURES,
    'shap_value' : listing_shap[0],
    'feature_value': listing.values[0]
}).sort_values('shap_value', key=abs, ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [TEAL if v > 0 else RED for v in contrib['shap_value']]

bars = ax.barh(
    contrib['feature'][::-1].str.replace('_',' ').str.title(),
    contrib['shap_value'][::-1],
    color=colors[::-1], edgecolor='white', height=0.7
)

ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('SHAP value (teal = pushed price UP, red = pushed price DOWN)')
ax.set_title(f'Why did the model predict €{pred_price:.0f} for this listing?\n'
             f'Top 10 contributing features',
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(REPORTS / 'fig_ml_shap_single.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Prediction for this listing: €{pred_price:.0f}/night')
print(f'Base (average) prediction : €{np.expm1(base_value):.0f}/night')
print()
print('Top factors pushing price UP:')
for _, row in contrib[contrib['shap_value'] > 0].head(3).iterrows():
    print(f'  {row["feature"]:<35}: +{row["shap_value"]:.4f}  (value={row["feature_value"]:.2f})')
print('\nTop factors pushing price DOWN:')
for _, row in contrib[contrib['shap_value'] < 0].head(3).iterrows():
    print(f'  {row["feature"]:<35}: {row["shap_value"]:.4f}  (value={row["feature_value"]:.2f})')

## Summary

| Finding | Business Implication |
|---------|---------------------|
| Neighbourhood is #1 driver | Location matters more than any property feature |
| Room type is #2 driver | Entire homes command a proven premium |
| Accommodates in top 5 | Larger properties can charge proportionally more |
| review_scores_rating matters | Better ratings translate to measurable price premium |
| Amenities have small but real impact | Adding kitchen/washer has quantified value |

**Charts saved:**
- fig_ml_shap_importance.png
- fig_ml_shap_beeswarm.png
- fig_ml_shap_dependence.png
- fig_ml_shap_single.png